In [1]:
import xarray as xr 
import tams
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import pandas as pd
import warnings
from joblib import Parallel, delayed
warnings.filterwarnings('ignore')

* Extracted from tams_predata.ipynb

* In order to take care of some nan values found in ctt, we're going to crop the lon coordinates as they are the main problem (see nan_check for more information)

In [2]:
root = "./"
pr_pre = "precipitation_IMERG_1999_WA" # This can be changed to match the data and resolution.
data_path_pr = f"{root}TFM/{pr_pre}.nc" # This is 0.1x0.1

temp_pre = "tb_merg_1999_WA" # This can be changed to match the data and resolution.
data_path_temp = f"{root}TFM/{temp_pre}_0.1x0.1.nc"

pr_data = xr.open_dataset(data_path_pr, chunks={'time':48, 'lat':-1,'lon':-1})
pr_data = pr_data.transpose("time", "lat", "lon") # In order to put dims correctly time,lat,lon.
pr_data = pr_data.rename({"precipitation":"pr"})

temp_data = xr.open_dataset(data_path_temp, chunks={'time':48, 'lat':-1,'lon':-1})
temp_data = temp_data.rename({"Tb":"ctt"})


# Interpolate in order to make merge faster for different resolutions
#pr_data = pr_data.interp_like(temp_data, method='nearest')

In [3]:
ds_full = xr.merge([temp_data, pr_data])

tbI = temp_data['ctt'].fillna(300)
tp = pr_data['pr']

- Let's fix the exact timestamp where we have a row full of nans. For now, I'll handle them as replacing them into 300 K but just that day and hour. 

In [26]:
ds_full

<xarray.Dataset> Size: 13GB
Dimensions:  (time: 5856, lat: 300, lon: 900)
Coordinates:
  * time     (time) datetime64[ns] 47kB 1999-06-01 ... 1999-09-30T23:30:00
  * lat      (lat) float32 1kB 0.05 0.15 0.25 0.35 ... 29.65 29.75 29.85 29.95
  * lon      (lon) float32 4kB -39.95 -39.85 -39.75 -39.65 ... 49.75 49.85 49.95
Data variables:
    ctt      (time, lat, lon) float32 6GB dask.array<chunksize=(124, 300, 900), meta=np.ndarray>
    pr       (time, lat, lon) float32 6GB dask.array<chunksize=(124, 300, 900), meta=np.ndarray>
Attributes: (12/13)
    CDI:             Climate Data Interface version 2.0.4 (https://mpimet.mpg...
    Conventions:     CF-1.6
    BeginDate:       1999-06-01
    BeginTime:       00:00:00.000Z
    EndDate:         1999-06-01
    EndTime:         00:59:59.999Z
    ...              ...
    InputPointer:    merg_1999060100_4km-pixel
    title:           NCEP/CPC 4km Global (60N - 60S) IR Dataset
    ProductionTime:  2025-08-16T01:55:40.709Z
    DOI:             10.5067/P4HZB9N27EKU
    history:         Tue Feb 17 18:29:38 2026: cdo remapcon2,/home/emohino/wo...
    CDO:             Climate Data Operators version 2.0.4 (https://mpimet.mpg...

In [4]:
cesI, _ = tams.identify(tbI, parallel=True)

ds = xr.Dataset({'tb': tbI, 'tp': tp})

def fun(ds_clice, ce):
    if not ce.empty:
        ce = tams.data_in_contours(ds_clice.tb, ce, agg=("mean", "min"), merge=True)
        ce = tams.data_in_contours(ds_clice.tp, ce, agg=("mean", "min"), merge=True)
    return ce

cesI_all = Parallel(n_jobs=2, verbose=10)(
    delayed(fun)(ds.isel(time=i).copy(deep=False), ce.copy())
    for i, ce in enumerate(cesI)
)

[Parallel(n_jobs=-2)]: Using backend LokyBackend with 7 concurrent workers.
[Parallel(n_jobs=-2)]: Done   4 tasks      | elapsed:   19.9s
[Parallel(n_jobs=-2)]: Done  11 tasks      | elapsed:   24.1s
[Parallel(n_jobs=-2)]: Done  18 tasks      | elapsed:   28.1s
[Parallel(n_jobs=-2)]: Done  27 tasks      | elapsed:   33.1s
[Parallel(n_jobs=-2)]: Done  36 tasks      | elapsed:   40.2s
[Parallel(n_jobs=-2)]: Done  47 tasks      | elapsed:   45.4s
[Parallel(n_jobs=-2)]: Done  58 tasks      | elapsed:   51.8s
[Parallel(n_jobs=-2)]: Done  71 tasks      | elapsed:   59.4s
[Parallel(n_jobs=-2)]: Done  84 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-2)]: Done  99 tasks      | elapsed:  1.3min
[Parallel(n_jobs=-2)]: Done 114 tasks      | elapsed:  1.5min
[Parallel(n_jobs=-2)]: Done 131 tasks      | elapsed:  1.8min
[Parallel(n_jobs=-2)]: Done 148 tasks      | elapsed:  2.0min
[Parallel(n_jobs=-2)]: Done 167 tasks      | elapsed:  2.1min
[Parallel(n_jobs=-2)]: Done 186 tasks      | elapsed:  2

In [8]:
times = ds.time.values

for i, ce in enumerate(cesI_all):
    if not ce.empty:
        ce['time'] = times[i]

In [12]:
cesI_all

[                                             geometry       area_km2  \
 0   POLYGON ((32.75 9.03182, 32.8 9.05, 32.83461 9...   45742.615173   
 1   POLYGON ((20.95 3.89, 25.15 4.33125, 25.1875 4...  573904.670068   
 2   POLYGON ((20.05 1.32241, 20.45 1.375, 20.51428...   15033.112996   
 3   POLYGON ((-8.75 3.88684, -8.45 3.92059, -8.416...   12029.379921   
 4   POLYGON ((-8.85 6.00385, -8.76429 6.05, -8.441...  145632.012101   
 5   POLYGON ((-9.85 4.75, -9.77105 5.05, -9.7431 5...   20356.686192   
 6   POLYGON ((-11.15 10.52083, -10.15 10.5375, -10...   34735.669659   
 7   POLYGON ((-11.25 2.3, -10.35 3.13846, -10.3418...   20715.579653   
 8   POLYGON ((-11.95 4.62, -11.85 4.68077, -11.79 ...   18694.877861   
 9   POLYGON ((-14.55 1.16429, -13.65 1.2, -13.55 1...   63897.981796   
 10  POLYGON ((-21.75 0.80667, -21.6881 0.85, -21.4...    8879.588383   
 11  POLYGON ((-30.75 0.05, -28.66905 0.05, -27 0.7...  360399.088247   
 
                                               in

In [13]:
ces_master = pd.concat(cesI_all, ignore_index=True)
ces_master = gpd.GeoDataFrame(ces_master, geometry='geometry')
ces_master.to_parquet('tams_cloud.parquet')
print("finish.")

finish.
